# Semantical annotations with Azure OpenAI client

In [1]:
import sys
sys.path.append("../")

In [2]:
import os
import openai
import sqlite3

from pandas import read_csv
from tqdm.auto import tqdm 
from configparser import ConfigParser

from llm_library.openai import configure_azure_client
from llm_library.prompt_templating import UserPrompt
from llm_library.prompt_templating import SystemPrompt
from llm_library.prompt_templating import AssistantPrompt
from llm_library.prompt_templating import InputList

## I. Set up Azure API 

In [3]:
config = ConfigParser()
status = config.read('../../model_configurations/azure_gpt-35-turbo.ini') 
assert status == ['../../model_configurations/azure_gpt-35-turbo.ini']

In [4]:
client = configure_azure_client(config)

## II. Annotate words with GPT  

In [5]:
PROMPT_PATH = 'prompts'
INPUT_FILE = 'data/soned_1000_margendatud.csv'

CHUNK_SIZE = 10
MODEL = config['azure-configuration']['model']

OUTPUT_FILE = f'results/run_01_{MODEL}_a.db'

In [6]:
lemmas = read_csv(INPUT_FILE, index_col=0).pipe(lambda df: df[~df['0'].isna()])['lemma']
assert len(lemmas) == 501, "Fix current state"

In [7]:
connection = sqlite3.connect(OUTPUT_FILE)
cursor = connection.cursor()
cursor.execute("DROP TABLE IF EXISTS  model_annotations")

# Create a table with the required columns
cursor.execute(
"""
CREATE TABLE model_annotations
(
    input TEXT,
    model TEXT,
    semantic_class TEXT,
    response TEXT
)
""")

connection.commit()

In [8]:
semantic_class = 'spatial_region'

print(f'Semantic class: {semantic_class}')
file_path = os.path.join(PROMPT_PATH, semantic_class, "prompt.txt")
with open(file_path, "r", encoding="utf-8") as f:
    prompt_text = f.read()
print('Prompt text')
print(prompt_text)
print('------------------------------------------------------------------------------------------')

# Process lemmas in chunk    
for i in tqdm(range(0, len(lemmas), CHUNK_SIZE)):
    chunk = lemmas[i:i + CHUNK_SIZE]

    # Query GPT
    try:
        response = client.chat.completions.create(
            model = config['azure-configuration']['deployment_name'],
            messages = [
                SystemPrompt(prompt_text),
                UserPrompt(InputList(chunk, format='line_separated'))
            ]
        )
    except Exception as e:
        # Store the cause of an abnormal execution
        cursor.execute(
        """
        INSERT INTO model_annotations (input, model, semantic_class, response)
        VALUES ('{input}', '{model}', '{semantic_class}', '{response}')
        """.format(
            input=InputList(chunk, format='line_separated'),
            model=MODEL,
            semantic_class=semantic_class,
            response='FAIL'
        ))
        continue
        

    # Store results of normal execution
    cursor.execute(
    """
    INSERT INTO model_annotations (input, model, semantic_class, response)
    VALUES ('{input}', '{model}', '{semantic_class}', '{response}')
    """.format(
        input=InputList(chunk, format='line_separated'),
        model=MODEL,
        semantic_class=semantic_class,
        response=response.choices[0].message.content
    ))
    connection.commit()

connection.close()

Semantic class: spatial_region
Prompt text
Provide the output in JSON format, where:

'1' indicates that the given Estonian word is a type of region. 
'0' indicates that the word does not meet these criteria.
Evaluate the following words accordingly where each word is on a separate line.
------------------------------------------------------------------------------------------


  0%|          | 0/51 [00:00<?, ?it/s]

In [9]:
print("We are done!")

We are done!
